# 🧠 EEG Signal Data Exploration & EDA

This notebook focuses on analyzing the raw data collected from the TGAM module. We aim to identify signal patterns, check for noise, and ensure that our features are separable for classification.

### Goals:
1. Visualize raw EEG bands per class.
2. Check for class imbalance.
3. Identify key features for command recognition.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add parent directory to path to import config
sys.path.append('..')
import config

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

## 1. Load Data
We use the `raw_data.csv` generated by `collect_data.py`.

In [ ]:
data_path = os.path.join('..', config.DATA_DIR, 'raw_data.csv')

if not os.path.exists(data_path):
    print(f"❌ Could not find {data_path}. Please run collect_data.py first!")
else:
    df = pd.read_csv(data_path)
    print(f"✅ Loaded {len(df)} samples.")
    display(df.head())

## 2. Class Distribution
Is our dataset balanced? We need enough samples for each command to train fairly.

In [ ]:
if 'label' in df.columns:
    # Map numeric labels to names for clarity
    class_names = {k: v for k, v in config.CLASSES.items()}
    df['class_name'] = df['label'].map(class_names)
    
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='class_name', palette='viridis')
    plt.title("Samples per Classification Class")
    plt.show()

## 3. Visualize Signal Trends
Let's check if high-level metrics like **Attention** and **Meditation** vary across states.

- **FORWARD** usually correlates with high **Attention**.
- **STOP** or **IDLE** might correlate with high **Meditation**.

In [ ]:
metrics = ['attention', 'meditation']
fig, axes = plt.subplots(len(metrics), 1, figsize=(12, 10))

for i, metric in enumerate(metrics):
    sns.boxplot(data=df, x='class_name', y=metric, ax=axes[i], palette='magma')
    axes[i].set_title(f"{metric.capitalize()} Distribution per Class")

plt.tight_layout()
plt.show()

## 4. EEG Band Analysis
Let's look at the brainwave bands (Delta, Theta, Alpha, Beta, Gamma).

**Blinks** (LEFT, RIGHT, STOP) usually cause massive spikes in **Delta** and **Theta** because of muscular artifacts.

In [ ]:
bands = ['delta', 'theta', 'lowAlpha', 'highAlpha', 'lowBeta', 'highBeta']

for band in bands:
    plt.figure(figsize=(12, 4))
    # Plotting first 300 samples of the band per class to see patterns
    for cls in df['class_name'].unique():
        sample = df[df['class_name'] == cls][band].reset_index(drop=True).head(300)
        plt.plot(sample, label=cls, alpha=0.7)
    
    plt.title(f"{band.capitalize()} Waveform Patterns (First 300 samples)")
    plt.legend()
    plt.xlabel("Sample Index (Relative)")
    plt.ylabel("Signal Power")
    plt.show()

## 5. Feature Correlation
Are some bands correlated? (e.g., Alpha waves often move together).

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[config.FEATURES].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()